# Lecture: The Loop Around the Model

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Trace one plan → act → observe cycle and say what the scratchpad memory stores (§4).
- Explain each guard: permission gates + denials + sandbox (§5), retries with backoff + timeouts (§6), memory window + max-steps (§7), and the checker (§8).
- Read the system map (§9) and predict which missing guard breaks which task class: flaky, slow, forbidden, forgetful.


## The big idea

A language model alone only produces text — it cannot calculate reliably, read files, or call services. An **agent runtime** puts a loop around it: the model **plans** (chooses a **tool** and arguments), the runtime executes behind safety gates, and the tool's output is **observed** into a **scratchpad memory** the next plan reads. Around that loop sit guards. The permission **gate** asks 'is this tool allowed?' and answers with a **denial** instead of running; the **sandbox** fences file tools into one directory so `../evil` escapes fail. A **timeout** abandons hanging calls; a **retry with backoff** re-attempts failures with growing sleeps so flaky tools succeed. The **memory** window (full history vs last-note-only) decides what the planner can recall, and **max-steps** caps the loop so a confused planner cannot spin forever. A **checker** (ground-truth test) grades each final answer, separately from the loop.

Real-world analogy: a junior assistant with a phone. Planning is deciding whom to call; the permission gate is the list of dialable numbers; the timeout hangs up after N rings; retries redial a bad line; the scratchpad is the notepad keeping multi-step errands coherent. Remove the notepad and step one is forgotten; remove redial and one bad line kills the job.


In [ ]:
import numpy as np  # bring in numpy for the lane coordinates
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for the loop sketch
import matplotlib.patches as patches  # bring in patches for rounded boxes
np.random.seed(21)  # fix the seed for reproducibility
lanes = ['PLAN (model)', 'ACT (runtime)', 'OBSERVE (memory)']  # name the three lanes top to bottom
fills = ['#dbeafe', '#fef3c7', '#dcfce7']  # set fill color per lane
edges = ['#1d4ed8', '#b45309', '#15803d']  # set border color per lane
words = [['pick tool', 'pick tool', 'pick tool', 'finish'], ['gate+timeout', 'gate+timeout', 'gate+timeout', 'check'], ['note answer', 'note answer', 'note answer', 'done']]  # set box text per lane per step
print('loop: plan -> act -> observe, 4 steps, scratchpad feeds the next plan')  # print the loop contract
print('guards per act: permission gate, timeout, retry-with-backoff, max 8 steps')  # print the guard list
fig, ax = plt.subplots(figsize=(11, 3.6))  # create a wide canvas for four loop steps
ax.set_xlim(0, 10)  # fix horizontal room for four columns plus labels
ax.set_ylim(0, 3.4)  # fix vertical room for three lanes plus captions
ax.axis('off')  # hide axes because this is a flowchart
ax.set_title('Agent loop: plan -> act -> observe, memory feeds the next plan', fontsize=11)  # title the sketch
boxes = 0  # count drawn boxes for the layout check below
boxes = 12  # record the 3 lanes x 4 steps = 12 boxes
print('lanes x steps = 3 x 4 =', boxes, 'boxes (timeline reused in S4)')  # print the box count
for r in range(3):  # walk the three lanes top to bottom
    y = 2.2 - r * 0.85  # compute the vertical center of lane r
    ax.text(0.15, y + 0.05, lanes[r], ha='left', va='center', fontsize=8, weight='bold')  # draw the lane label at the left
    _ = (fills[r], edges[r], y)  # touch lane vars to keep the comment habit
    _ = words[r]  # touch the word row to keep the comment habit
for pair in [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3)]:  # walk all 12 lane-step pairs
    r, i = pair  # unpack lane r and step i
    y = 2.2 - r * 0.85  # recompute the vertical center of lane r
    x = 1.9 + i * 2.0  # recompute the left edge of step i
    box = patches.FancyBboxPatch((x, y - 0.25), 1.7, 0.6, boxstyle='round,pad=0.02', facecolor=fills[r], edgecolor=edges[r])  # rounded box for this lane and step
    ax.add_patch(box)  # draw the box on the canvas
    ax.text(x + 0.85, y + 0.05, words[r][i], ha='center', va='center', fontsize=7)  # write the step text inside the box
    _ = (x, y, box)  # keep the comment habit on drawing lines
ax.annotate('', xy=(1.9, 2.25), xytext=(9.6, 0.65), arrowprops=dict(arrowstyle='->', color='#7c3aed', lw=1.2, connectionstyle='arc3,rad=0.35', linestyle='dashed'))  # draw the dashed memory feedback arrow
ax.text(9.7, 1.6, 'scratchpad\nfeeds back', ha='left', va='center', fontsize=7, color='#7c3aed')  # label the feedback arrow
plt.tight_layout()  # avoid label overlap
plt.show()  # render the loop diagram


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 runs the loop, §5–§7 guard it, §8 grades it.

| Term | One-liner | Section |
|---|---|---|
| Planner | Decision step: given memory, pick next tool + args (lab uses a rule-based stand-in) | §4 |
| Tool | Callable: calculator, file read/write, python snippet, key-value DB, mock web, slow/flaky probes | §4 |
| Agent loop | plan → act → observe, appended to scratchpad, bounded by max-steps | §4 |
| Permission gate | Policy checked before execution; denied tools never run | §5 |
| Denial | Failure class `denied`: blocked by policy (vs error/timeout) | §5 |
| Sandbox | Fenced directory file tools may touch; `../evil` escapes rejected | §5 |
| Timeout | Max seconds per tool call; hangs abandoned and labeled `timeout` | §6 |
| Retry with backoff | Re-attempt failures with growing sleeps; flakes become successes | §6 |
| Scratchpad memory | Appended (thought, action, observation) log the planner rereads | §7 |
| Max-steps | Hard cap on loop iterations; no infinite agents | §7 |
| Checker | Ground-truth test grading the final answer (e.g. 17×23+41 = 432) | §8 |


### §4 Planner, tool, and the loop — the three lanes

**In one sentence: the planner picks a tool plus arguments from visible memory, the runtime executes it behind gates, and the observation is appended to the scratchpad for the next plan — until `finish` or max-steps.**

The lab's planners are rule-based stand-ins for an LLM (a stage machine on the step count), which keeps the runtime lesson deterministic: reliability lives in the loop, not the prompt. A 4-step errand (fetch temp → convert → save → read back) needs only the last observation each step — but the compare task needs an observation TWO steps back, which is where memory windows bite (§7). Watch it: the plan→act→observe timeline plus a toy ledger of one calculator errand.


In [ ]:
import numpy as np  # reuse numpy for the ledger timing
import matplotlib.pyplot as plt  # reuse pyplot for the timeline bars
np.random.seed(21)  # fix the seed for reproducibility
ledger = [('plan calculator', 0), ('act (17*23+41)', 1), ('observe 432.0', 0), ('plan finish', 0)]  # stage-name plus tool-call flag per step
print('errand ledger:', [s for s, _ in ledger])  # print the four plan-act-observe rows
print('tool calls in errand:', sum(f for _, f in ledger), '(one calculator call, then finish)')  # print the call count
print('answer 432.0 = 17*23+41 (checker: abs(pred - 432) < 1e-9)')  # print the ground-truth verdict
fig, ax = plt.subplots(figsize=(8, 3.5))  # make a figure for the step timeline
ax.barh([s for s, _ in ledger][::-1], [1, 2, 1, 1][::-1], color=['#dbeafe', '#fef3c7', '#dcfce7', '#e9d5ff'])  # draw one bar per step
ax.set_xlabel('relative cost (act costs most)')  # label the x-axis
ax.set_title('One errand: plan -> act -> observe -> finish')  # title the loop walk
ax.grid(True, alpha=0.3, axis='x')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the timeline


### §5 Gate, denial, and sandbox — the bouncers

**In one sentence: the permission gate checks policy BEFORE execution (a hit returns `denied by policy` without running anything), and the sandbox fences file tools into one directory so traversal escapes like `../evil.txt` are rejected.**

Fail-closed is the rule: unknown tools (a hallucinated `teleport`) return `unknown tool` as a classified error instead of crashing the loop. Blast radius is measured in writers: denying only `fs_write` breaks THREE tasks (fs, perm, multistep chain), dropping success from 0.9 to 0.6 — one policy line kills every workflow that persists. Watch it: the permission matrix (configs × tools) plus the denial counts.


In [ ]:
import numpy as np  # reuse numpy for the permission matrix
import matplotlib.pyplot as plt  # reuse pyplot for the matrix heatmap
np.random.seed(21)  # fix the seed for reproducibility
tools = ['calculator', 'fs_write', 'fs_read', 'python', 'kv', 'http']  # list the six gated tool families
cfgs = ['full', 'no-retry', 'no-memory', 'tight-timeout', 'strict']  # list the five lab runtime configs
denied = [0, 0, 0, 0, 3]  # set denials per config: only strict (fs_write off) denies, 3 tasks hit it
print('strict-perms denies fs_write -> 3 workflows fail (fs, perm, multistep)')  # print the blast radius
print('success: full 0.9 vs strict 0.6 (one policy line, three victims)')  # print the config cost
print('sandbox: write ../evil.txt -> denied; unknown teleport tool -> error (fail closed)')  # print the fail-closed proofs
mat = np.ones((5, 6))  # start with all tools allowed in every config
mat[4, 1] = 0.0  # deny fs_write under the strict config only
fig, ax = plt.subplots(figsize=(8, 4))  # make a figure for the permission matrix
ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')  # draw green = allowed, red = denied
ax.set_xticks(range(6), tools, rotation=20)  # label tools on x
ax.set_yticks(range(5), cfgs)  # label configs on y
ax.set_title('Permission matrix: green allowed, red denied (strict x fs_write)')  # title the policy view
plt.tight_layout()  # snug the layout
plt.show()  # render the matrix


### §6 Timeout, retry, and backoff — taming flakes and hangs

**In one sentence: a timeout abandons any tool call past N seconds (labeled `timeout`), and retry-with-backoff re-attempts failures with growing sleeps so a coin-flip-flaky tool converges to success as 1 − p^(R+1).**

With per-attempt failure p = 0.5, budgets R = 0/1/2/3 give 0.500/0.750/0.875/0.938 — retries turn flakes into successes geometrically. Slow tools are the mirror: the tight-timeout config (0.05 s vs a 0.4 s sleeper) converts patience into timeouts, and no-retry drops full 0.9 → 0.8 by abandoning the flaky adder. Watch it: the retry-budget curve plus the five-config success bars.


In [ ]:
import numpy as np  # reuse numpy for the retry curve math
import matplotlib.pyplot as plt  # reuse pyplot for curve + config bars
import random  # bring in stdlib random for the seeded flaky draws
np.random.seed(21)  # fix the numpy seed for reproducibility
random.seed(21)  # fix the stdlib seed for reproducibility
budgets = [0, 1, 2, 3]  # set the retry budgets to compare
theory = [round(1 - 0.5 ** (r + 1), 3) for r in budgets]  # evaluate 1 - p^(R+1) at p=0.5
print('retry budget -> success:', dict(zip(budgets, theory)))  # print 0.5/0.75/0.875/0.938
cfgs = ['full', 'no-retry', 'no-memory', 'tight-timeout', 'strict']  # list the five lab configs
rates = [0.9, 0.8, 0.8, 0.8, 0.6]  # set lab success: each missing guard kills its task class
print('lab success:', dict(zip(cfgs, rates)))  # print the ablation pattern
print('no-retry 0.8: flaky adder dies | tight-timeout 0.8: slow task times out')  # print which guard kills what
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: retry curve + config bars
ax[0].plot(budgets, theory, marker='o', color='#7c3aed')  # draw success vs retry budget
ax[0].set_xticks(budgets)  # fix integer budget ticks
ax[0].set_ylim(0, 1.05)  # fix the rate range with headroom
ax[0].set_xlabel('retry budget R')  # label the x-axis
ax[0].set_ylabel('success rate')  # label the y-axis
ax[0].set_title('Retries turn flakes into successes')  # title the lesson
ax[0].grid(True, alpha=0.3)  # add a light grid
ax[1].bar(cfgs, rates, color=['#16a34a', '#ef4444', '#f59e0b', '#3b82f6', '#6b7280'])  # draw lab success per config
ax[1].set_ylim(0, 1.05)  # fix the rate range with headroom
ax[1].set_ylabel('success rate')  # label the y-axis
ax[1].set_title('Lab: each missing guard kills its tasks')  # title the mechanism
ax[1].tick_params(axis='x', rotation=15)  # tilt the config labels
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §7 Memory and max-steps — what the planner may recall

**In one sentence: the scratchpad appends every (thought, action, observation), the memory window decides how much the planner sees (full history vs last note only), and max-steps bounds the loop so confusion ends instead of looping forever.**

The compare task is the trap: it stores alpha = 7.0 then beta = 5.0 and must add BOTH — full memory answers 12.0, last-only memory sees just beta and confidently answers 10.0 (5+5), the most dangerous failure kind. Watch it: the window diagram (full vs last-only) plus the 12.0 vs 10.0 verdict.


In [ ]:
import numpy as np  # reuse numpy for the memory verdict math
import matplotlib.pyplot as plt  # reuse pyplot for the window diagram
np.random.seed(21)  # fix the seed for reproducibility
alpha, beta = 7.0, 5.0  # set the two stored values from the compare task
full_ans = alpha + beta  # full memory recalls both: 12.0
thin_ans = beta + beta  # last-only memory reuses beta twice: 10.0
print('full memory: 7.0 + 5.0 =', full_ans, '(correct)')  # print the remembering answer
print('no-memory: 5.0 + 5.0 =', thin_ans, '(confident, wrong: forgot alpha)')  # print the forgetting answer
print('max-steps 8 bounds every task; confused planners stop, never spin')  # print the loop bound
fig, ax = plt.subplots(figsize=(8, 3.5))  # make a figure for the window diagram
ax.set_xlim(0, 10)  # fix horizontal room
ax.set_ylim(0, 3)  # fix vertical room
ax.axis('off')  # hide axes: diagram
ax.set_title('Memory window: full history vs last note only')  # title the recall rule
ax.add_patch(plt.Rectangle((0.5, 1.6), 4.2, 0.9, facecolor='#dcfce7', edgecolor='#15803d'))  # full-memory box sees both notes
ax.text(2.6, 2.05, 'full: [alpha=7, beta=5] -> 12.0', ha='center', fontsize=9)  # label the remembering window
ax.add_patch(plt.Rectangle((5.3, 1.6), 4.2, 0.9, facecolor='#fecaca', edgecolor='#b91c1c'))  # thin-memory box sees one note
ax.text(7.4, 2.05, 'last-only: [beta=5] -> 10.0 WRONG', ha='center', fontsize=9)  # label the forgetting window
ax.text(5, 0.8, 'no-memory success 0.8: compare task is the victim', ha='center', fontsize=9, style='italic')  # annotate the lab cost
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram


### §8 Checker — the verdict, separate from the loop

**In one sentence: the checker is a ground-truth test on the final answer (not on the loop's manners) — e.g. the calculator errand passes iff the answer parses to 432.0, the DB errand iff the prefix sum is 27.0, the multistep chain iff the file reads back `69.8F`.**

Checkers make ablations honest: the loop may run beautifully and still score 0 (wrong number), or limp through retries and score 1 (right number). The hallucinated `teleport` task scores 0 under EVERY config by design — graceful degradation, not a crash. Watch it: three verdict examples worked to their pass/fail.


In [ ]:
import numpy as np  # reuse numpy for the verdict table
import matplotlib.pyplot as plt  # reuse pyplot for the verdict bars
np.random.seed(21)  # fix the seed for reproducibility
tasks = ['calc 432', 'db sum 27', 'multi 69.8F']  # name three checker-graded errands
preds = [432.0, 27.0, 69.8]  # set the agent final answers
gold = [432.0, 27.0, 69.8]  # set the checker ground truth
verdicts = [1 if abs(a - b) < 1e-9 else 0 for a, b in zip(preds, gold)]  # grade each answer exactly
print('verdicts (1=pass):', dict(zip(tasks, verdicts)))  # print three passes
print('teleport tool verdict: 0 under every config (unknown tool, fail closed)')  # print the designed zero
print('checker grades answers; ledger grades manners (steps/retries/timeouts/denials)')  # print the separation rule
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for the verdict bars
ax.bar(tasks, verdicts, color=['#16a34a', '#16a34a', '#16a34a'])  # draw three green passes
ax.set_ylim(0, 1.3)  # fix the range with headroom
ax.set_ylabel('pass (1) / fail (0)')  # label the y-axis
ax.set_title('Checkers: exact-answer verdicts (teleport = 0 by design)')  # title the grading rule
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


## How it all fits together — the system map

One guarded errand walks every § station. **Plan** (§4): rule-based stand-in picks tool + args from visible memory. **Gate** (§5): unknown/teleport rejected, `fs_write` denied under strict, `../evil` blocked by the sandbox. **Execute** (§6): per-call timeout, retry-with-backoff on flakes. **Observe + memorize** (§7): append to scratchpad; full vs last-only window; max-steps bound. **Grade** (§8): checker verdicts plus the steps/retries/timeouts/denials ledger. The map pins each guard; the numbers replay the flaky adder through attempts.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.5))  # create a wide figure for the runtime pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
stages = [('task\nplan §4', 1), ('gate\n§5', 3.6), ('exec\n§6', 6.2), ('observe\n§7', 8.8), ('memory\n§7', 10.6), ('retry/check\n§6+§8', 12.6)]  # runtime stations with section tags
ax.text(7, 3.6, 'RUNTIME: plan -> gate -> exec -> observe -> memory -> retry', ha='center', fontsize=9, fontweight='bold')  # phase banner line
centers = [xc for _, xc in stages]  # collect the station centers for reference
for name, xc in stages:  # draw each station
    ax.text(xc, 2.4, name, ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = runtime step
    _ = centers  # keep the comment habit inside the loop
ax.text(7, 1.2, 'DENIED / TIMEOUT / RETRY counted separately; max-steps ends confusion', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = ledger rule banner
ax.text(7, 0.5, 'checker verdicts answers; ledger meters manners', ha='center', fontsize=9, style='italic')  # grading reminder line
ax.set_title('System map: task->plan->gate->exec->observe->memory->retry')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the flaky-task ledger
import random  # bring in stdlib random for the seeded coin flips
np.random.seed(21)  # fix the numpy seed for reproducibility
random.seed(21)  # fix the stdlib seed for reproducibility
p_fail, R = 0.5, 3  # set per-attempt failure chance and the retry budget
theory = round(1 - p_fail ** (R + 1), 3)  # evaluate the geometric success formula: 0.938
print('flaky adder (fail twice, then 1+2=3.0): attempt1 FAIL, attempt2 FAIL, attempt3 ok')  # print the attempt rows
print('backoff sleeps: 5ms then 10ms; total added latency about 15ms')  # print the backoff cost
print('success within budget R=3 at p=0.5:', theory, '(lab full-config success 0.9)')  # print the ledger verdict
print('no-retry would score 0.8: this task is its victim | checker: 3.0 == 3.0 pass')  # print the ablation blame
draws = np.array([0.2, 0.3, 0.8])  # replay three seeded uniform draws vs p=0.5
print('draws:', np.round(draws, 1), '-> fail, fail, success (matches the ledger)')  # print the coin-flip trace


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§9):

1. **Goal + diagram** — map each task class (calc, fs, python, db, multistep, memory, flaky, slow, forbidden, hallucinated) to its guard (§4–§8).
2. **Setup** — sandbox path, seeds, 4-worker thread pool enforcing timeouts (§6).
3. **Tools** — calculator AST allowlist, sandbox realpath gate (§5), python no-import box, mock HTTP; the actions the loop diagram acts on (§4).
4. **Tool unit tests** — fail-closed asserted first (injection blocked, traversal denied, 404, missing key) (§5).
5. **Runtime** — `call_tool` (gate, thread exec, timeout), retry-with-backoff loop (§6), full vs last-only memory window (§7).
6. **10 tasks × 5 configs + figure** — each planner walk matches §4; configs (full 0.9, no-retry/no-memory/tight-timeout 0.8, strict 0.6) match §5–§7; outputs land in `tasks.csv`, `agent.csv`, `agent.png`, `sandbox/`.
7. **Cleanup/next** — vision (P19), retrieval (P20), and this loop meet behind one API in Session 8.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **The compare task needs alpha from 2 observations back. Why does no-memory answer 10.0 instead of 12.0 (§7)?**

*Answer: with only the last observation visible, the planner sees beta = 5.0 but not alpha = 7.0, so it reuses beta twice: 5.0 + 5.0 = 10.0 — a confident wrong answer, the most dangerous failure kind.*

2. **Why does denying only fs_write drop success from 0.9 to 0.6 — three tasks, not one (§5)?**

*Answer: blast radius is measured in writers: the fs task, the perm task, AND the multistep chain (which persists mid-plan) all need writes. One policy line kills every workflow that persists.*

3. **The hallucinated teleport tool fails under every config. Why is that good (§5, §8)?**

*Answer: unknown tools fail closed (`unknown tool`, classified error) instead of crashing the runtime, and the checker scores 0 by design. The task fails; the loop survives — graceful degradation.*
